<p align="right"><a href="./C2_W2_Assignment.ipynb">English</a> | <strong>中文</strong></p>

# Practice Lab: 手写数字识别的神经网络——多分类（Neural Networks for Handwritten Digit Recognition, Multiclass）

在本练习中，你将用一个神经网络识别手写数字 0-9。

# 大纲
- [ 1 - 库 Packages ](#1)
- [ 2 - ReLU 激活](#2)
- [ 3 - Softmax 函数](#3)
  - [ 练习 Exercise 1](#ex01)
- [ 4 - 神经网络 Neural Networks](#4)
  - [ 4.1 问题描述](#4.1)
  - [ 4.2 数据集](#4.2)
  - [ 4.3 模型表示](#4.3)
  - [ 4.4 Tensorflow 模型实现](#4.4)
  - [ 4.5 Softmax 的位置](#4.5)
    - [ 练习 Exercise 2](#ex02)

_**注意：** 为避免自动评分器（autograder）出错，你不能编辑或删除本 notebook 中的非评分 cell，也请不要新增任何 cell。
**当你通过本作业后**，如果想对其中的非评分代码做实验，可以按 notebook 底部的说明操作。_

<a name="1"></a>
## 1 - 库 Packages

首先，运行下面的 cell 导入本作业需要的所有库。
- [numpy](https://numpy.org/) 是 Python 中科学计算的基础库。
- [matplotlib](http://matplotlib.org) 是 Python 中常用的绘图库。
- [tensorflow](https://www.tensorflow.org/) 是一个常用的机器学习平台。

In [59]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.activations import linear, relu, sigmoid
%matplotlib widget
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')

import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

from public_tests import * 

from autils import *
from lab_utils_softmax import plt_softmax
np.set_printoptions(precision=2)

<a name="2"></a>
## 2 - ReLU 激活
本周引入了一种新的激活——修正线性单元（Rectified Linear Unit，ReLU）。
$$ a = max(0,z) \quad\quad\text {# ReLU function} $$

In [60]:
plt_act_trio()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

<img align="right" src="./images/C2_W2_ReLu.png"     style=" width:380px; padding: 10px 20px; " >
右边课程里的例子展示了 ReLU 的一个应用。此例中，导出的 "awareness" 特征不是二值的，而是有连续的取值范围。sigmoid 最适合 on/off 或二值情形，而 ReLU 提供一种连续的线性关系；此外它还有一段输出为零的 'off' 区间。
这个 "off" 特性使 ReLU 成为一个非线性激活。为什么需要它？它让多个 unit 能够互不干扰地为最终函数做贡献。这一点在配套的选修实验里有更多考察。

<a name="3"></a>
## 3 - Softmax 函数
一个多分类神经网络生成 N 个输出，从中选出一个作为预测答案。在输出层，由一个线性函数生成向量 $\mathbf{z}$，送入 softmax 函数。softmax 函数把 $\mathbf{z}$ 转换成一个概率分布（如下所述）。施加 softmax 后，每个输出都在 0 到 1 之间、且所有输出之和为 1，可解释为概率。送入 softmax 的输入越大，对应的输出概率越大。
<center>  <img  src="./images/C2_W2_NNSoftmax.PNG" width="600" />  

softmax 函数可写作：
$$a_j = \frac{e^{z_j}}{ \sum_{k=0}^{N-1}{e^{z_k} }} \tag{1}$$

其中 $z = \mathbf{w} \cdot \mathbf{x} + b$，N 是输出层的特征/类别数。

<a name="ex01"></a>
### 练习 Exercise 1
我们来写一个 NumPy 实现：

In [61]:
# UNQ_C1
# GRADED CELL: my_softmax

def my_softmax(z):  
    """ Softmax converts a vector of values to a probability distribution.
    Args:
      z (ndarray (N,))  : input data, N features
    Returns:
      a (ndarray (N,))  : softmax of z
    """    
    ### START CODE HERE ### 
    ez = np.exp(z)
    a = ez/np.sum(ez)
    ### END CODE HERE ### 
    return a

In [62]:
z = np.array([1., 2., 3., 4.])
a = my_softmax(z)
atf = tf.nn.softmax(z)
print(f"my_softmax(z):         {a}")
print(f"tensorflow softmax(z): {atf}")

# BEGIN UNIT TEST  
test_my_softmax(my_softmax)
# END UNIT TEST  

my_softmax(z):         [0.03 0.09 0.24 0.64]
tensorflow softmax(z): [0.03 0.09 0.24 0.64]
 All tests passed.


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    一种实现用一个 for 循环先构建分母，再用第二个循环计算每个输出。

```python
def my_softmax(z):
    N = len(z)
    a =                     # initialize a to zeros
    ez_sum =                # initialize sum to zero
    for k in range(N):      # loop over number of outputs
        ez_sum +=           # sum exp(z[k]) to build the shared denominator
    for j in range(N):      # loop over number of outputs again
        a[j] =              # divide each the exp of each output by the denominator
    return(a)
```
<details>
  <summary><font size="3" color="darkgreen"><b>点击查看代码</b></font></summary>

```python
def my_softmax(z):
    N = len(z)
    a = np.zeros(N)
    ez_sum = 0
    for k in range(N):
        ez_sum += np.exp(z[k])
    for j in range(N):
        a[j] = np.exp(z[j])/ez_sum
    return(a)

Or, a vector implementation:

def my_softmax(z):
    ez = np.exp(z)
    a = ez/np.sum(ez)
    return(a)

```

下面，改变 `z` 输入的值。特别注意分子里的指数如何放大数值间的微小差异，也注意各输出值之和为一。

In [63]:
plt.close("all")
plt_softmax(my_softmax)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

<a name="4"></a>
## 4 - 神经网络 Neural Networks

上周的作业里，你实现了一个做二分类的神经网络。本周你将把它扩展到多分类，这会用到 softmax 激活。

<a name="4.1"></a>
### 4.1 问题描述

在本练习中，你将用一个神经网络识别十个手写数字 0-9。这是一个从 n 个选项中选一个的多分类任务。手写数字的自动识别如今被广泛使用——从识别信封上的邮政编码，到识别银行支票上填写的金额。

<a name="4.2"></a>
### 4.2 数据集

你先加载本任务的数据集。
- 下面的 `load_data()` 函数把数据加载到变量 `X` 和 `y` 中

- 数据集包含 5000 个手写数字的训练样本 $^1$。

    - 每个训练样本是该数字的一张 20 像素 x 20 像素的灰度图。
        - 每个像素用一个浮点数表示，代表该位置的灰度强度。
        - 这 20×20 的像素网格被 "展开" 成一个 400 维向量。
        - 每个训练样本成为数据矩阵 `X` 中的一行。
        - 于是我们得到一个 5000 x 400 的矩阵 `X`，每行是一张手写数字图像的训练样本。

$$X =
\left(\begin{array}{cc}
--- (x^{(1)}) --- \\
--- (x^{(2)}) --- \\
\vdots \\
--- (x^{(m)}) ---
\end{array}\right)$$

- 训练集的第二部分是一个 5000 x 1 维的向量 `y`，包含训练集的标签
    - 图像是数字 `0` 时 `y = 0`，图像是数字 `4` 时 `y = 4`，以此类推。

$^1$<sub> 这是 MNIST 手写数字数据集 (http://yann.lecun.com/exdb/mnist/) 的一个子集</sub>

In [64]:
# load dataset
X, y = load_data()

#### 4.2.1 查看变量
我们更熟悉一下数据集。
- 一个好的起点就是把每个变量打印出来，看看它包含什么。

下面的代码打印变量 `X` 和 `y` 的第一个元素。

In [65]:
print ('The first element of X is: ', X[0])

The first element of X is:  [ 0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
  0.00e+00  0.00e+00  0.00e+00  0.00e+00  8.56e-06  1.94e-06 -7.37e-04
 -8.13e-03 -1.86e-02 -1.87e-02 -1.88e-02 -1.91e-02 -1.64e-02 -3.78e-03
  3.30e-04  1.28e-05  0.00e+00  0.00e+00  0.00e+00  0.00e+00  0.00e+00
  0.00e+00  0.00e+00  1.16e-04  1.20e-04 -1.40e-02 -2.85e-02  8.04e-02
  2.67e-01  2.74e-01  2.79e-01  2.74e-01  2.25e-0

In [66]:
print ('The first element of y is: ', y[0,0])
print ('The last element of y is: ', y[-1,0])

The first element of y is:  0
The last element of y is:  9


#### 4.2.2 查看变量的维度

熟悉数据的另一个方式是查看它的维度。请打印 `X` 和 `y` 的 shape，看看数据集里有多少个训练样本。

In [67]:
print ('The shape of X is: ' + str(X.shape))
print ('The shape of y is: ' + str(y.shape))

The shape of X is: (5000, 400)
The shape of y is: (5000, 1)


#### 4.2.3 可视化数据

你先可视化训练集的一个子集。
- 下面的 cell 中，代码从 `X` 随机选 64 行，把每行映射回一张 20x20 像素的灰度图，并把这些图一起显示出来。
- 每张图的标签显示在图的上方

In [68]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# You do not need to modify anything in this cell

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(5,5))
fig.tight_layout(pad=0.13,rect=[0, 0.03, 1, 0.91]) #[left, bottom, right, top]

#fig.tight_layout(pad=0.5)
widgvis(fig)
for i,ax in enumerate(axes.flat):
    # Select random indices
    random_index = np.random.randint(m)
    
    # Select rows corresponding to the random indices and
    # reshape the image
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # Display the image
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # Display the label above the image
    ax.set_title(y[random_index,0])
    ax.set_axis_off()
    fig.suptitle("Label, image", fontsize=14)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

<a name="4.3"></a>
### 4.3 模型表示

本作业你将使用的神经网络如下图所示。
- 它有两个用 ReLU 激活的 dense 层，后接一个用 linear 激活的输出层。
    - 回忆一下，我们的输入是数字图像的像素值。
    - 由于图像大小为 $20\times20$，这给我们 $400$ 个输入

<img src="images/C2_W2_Assigment_NN.png" width="600" height="450">

- 这些参数的维度是按"第 1 层 25 个 unit、第 2 层 15 个 unit、第 3 层 10 个输出 unit（每个数字一个）"的网络来定的。

    - 回忆一下，这些参数的维度是这样确定的：
        - 如果网络某层有 $s_{in}$ 个 unit、下一层有 $s_{out}$ 个 unit，那么
            - $W$ 的维度为 $s_{in} \times s_{out}$。
            - $b$ 是一个含 $s_{out}$ 个元素的向量

    - 因此，`W` 和 `b` 的形状为
        - layer1：`W1` 形状 (400, 25)，`b1` 形状 (25,)
        - layer2：`W2` 形状 (25, 15)，`b2` 形状 (15,)
        - layer3：`W3` 形状 (15, 10)，`b3` 形状 (10,)
>**注意：** 偏置向量 `b` 可以表示为一维 (n,) 或二维 (n,1) 数组。Tensorflow 用一维表示，本实验也保持这个约定：

<a name="4.4"></a>
### 4.4 Tensorflow 模型实现

Tensorflow 模型是一层层构建的。某层的输入维度（上面的 $s_{in}$）会自动为你算出。你指定某层的 *输出维度*，它决定了下一层的输入维度。第一层的输入维度由下面 `model.fit` 中指定的输入数据大小推导得出。
>**注意：** 也可以加一个输入层来指定第一层的输入维度，例如：
`tf.keras.Input(shape=(400,)),    #specify input shape`
我们这里会加上它，以说明模型大小的确定。

<a name="4.5"></a>
### 4.5 Softmax 的位置
如课程和 softmax 选修实验所述，训练时把 softmax 与损失函数合在一起（而非放在输出层），数值稳定性更好。这对 *构建* 模型和 *使用* 模型都有影响。
构建时：
* 最后一个 Dense 层应使用 'linear' 激活，这实际上就是不加激活。
* `model.compile` 语句通过加入 `from_logits=True` 来表明这一点。
`loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True) `
* 这不影响目标（target）的形式。对 SparseCategorialCrossentropy，目标是期望的数字 0-9。

使用模型时：
* 输出不是概率。如果想要输出概率，应用一个 softmax 函数。

<a name="ex02"></a>
### 练习 Exercise 2

下面，用 Keras 的 [Sequential 模型](https://keras.io/guides/sequential_model/) 和带 ReLU 激活的 [Dense 层](https://keras.io/api/layers/core_layers/dense/) 构建上面描述的三层网络。

In [69]:
# UNQ_C2
# GRADED CELL: Sequential model
tf.random.set_seed(1234) # for consistent results
model = Sequential(
    [               
        ### START CODE HERE ### 
        tf.keras.layers.InputLayer((400,)),
        tf.keras.layers.Dense(25, activation="relu", name="L1"),
        tf.keras.layers.Dense(15, activation="relu", name="L2"),
        tf.keras.layers.Dense(10, activation="linear", name="L3")
        ### END CODE HERE ### 
    ], name = "my_model" 
)

In [70]:
model.summary()

Model: "my_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 L1 (Dense)                  (None, 25)                10025     
                                                                 
 L2 (Dense)                  (None, 15)                390       
                                                                 
 L3 (Dense)                  (None, 10)                160       
                                                                 
Total params: 10,575
Trainable params: 10,575
Non-trainable params: 0
_________________________________________________________________


<details>
  <summary><font size="3" color="darkgreen"><b>期望输出（点击展开）</b></font></summary>
`model.summary()` 函数显示模型的一个有用摘要。注意，层的名字可能不同，因为除非指定，否则它们是自动生成的。

```
Model: "my_model"
_________________________________________________________________
Layer (type)                 Output Shape              Param #
=================================================================
L1 (Dense)                   (None, 25)                10025
_________________________________________________________________
L2 (Dense)                   (None, 15)                390
_________________________________________________________________
L3 (Dense)                   (None, 10)                160
=================================================================
Total params: 10,575
Trainable params: 10,575
Non-trainable params: 0
_________________________________________________________________
```

<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>

```python
tf.random.set_seed(1234)
model = Sequential(
    [
        ### START CODE HERE ###
        tf.keras.Input(shape=(400,)),     # @REPLACE
        Dense(25, activation='relu', name = "L1"), # @REPLACE
        Dense(15, activation='relu',  name = "L2"), # @REPLACE
        Dense(10, activation='linear', name = "L3"),  # @REPLACE
        ### END CODE HERE ###
    ], name = "my_model"
)
```

In [71]:
# BEGIN UNIT TEST     
test_model(model, 10, 400)
# END UNIT TEST     

All tests passed!


summary 里显示的参数数量，对应权重和偏置数组中元素的个数，如下所示。

我们进一步查看权重，验证 tensorflow 产生的维度与我们上面计算的一致。

In [72]:
[layer1, layer2, layer3] = model.layers

In [73]:
#### Examine Weights shapes
W1,b1 = layer1.get_weights()
W2,b2 = layer2.get_weights()
W3,b3 = layer3.get_weights()
print(f"W1 shape = {W1.shape}, b1 shape = {b1.shape}")
print(f"W2 shape = {W2.shape}, b2 shape = {b2.shape}")
print(f"W3 shape = {W3.shape}, b3 shape = {b3.shape}")

W1 shape = (400, 25), b1 shape = (25,)
W2 shape = (25, 15), b2 shape = (15,)
W3 shape = (15, 10), b3 shape = (10,)


**期望输出**
```
W1 shape = (400, 25), b1 shape = (25,)
W2 shape = (25, 15), b2 shape = (15,)
W3 shape = (15, 10), b3 shape = (10,)
```

下面的代码：
* 定义一个损失函数 `SparseCategoricalCrossentropy`，并通过加 `from_logits=True` 表明 softmax 应纳入损失计算
* 定义一个优化器。一个常用选择是自适应矩估计（Adaptive Moment，Adam），课程里讲过。

In [74]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
)

history = model.fit(
    X,y,
    epochs=40
)

Epoch 1/40
157/157 [==============================] - 1s 2ms/step - loss: 1.7094
Epoch 2/40
157/157 [==============================] - 0s 2ms/step - loss: 0.7480
Epoch 3/40
157/157 [==============================] - 0s 2ms/step - loss: 0.4428
Epoch 4/40
157/157 [==============================] - 0s 2ms/step - loss: 0.3463
Epoch 5/40
157/157 [==============================] - 0s 2ms/step - loss: 0.2977
Epoch 6/40
157/157 [==============================] - 0s 2ms/step - loss: 0.2630
Epoch 7/40
157/157 [==============================] - 0s 2ms/step - loss: 0.2361
Epoch 8/40
157/157 [==============================] - 0s 2ms/step - loss: 0.2131
Epoch 9/40
157/157 [==============================] - 0s 2ms/step - loss: 0.2004
Epoch 10/40
157/157 [==============================] - 0s 2ms/step - loss: 0.1805
Epoch 11/40
157/157 [==============================] - 0s 2ms/step - loss: 0.1692
Epoch 12/40
157/157 [==============================] - 0s 2ms/step - loss: 0.1580
Epoch 13/40
157/157 [====

#### Epoch 与 batch
上面的 `fit` 语句里 `epochs` 设为 40，表示整个数据集在训练中被应用 40 遍。训练时，你会看到描述训练进度的输出，形如：
```
Epoch 1/40
157/157 [==============================] - 0s 1ms/step - loss: 2.2770
```
第一行 `Epoch 1/40` 表示模型当前在跑第几个 epoch。为提高效率，训练数据集被拆成一个个 'batch'。Tensorflow 中 batch 的默认大小是 32。我们的数据集有 5000 个样本，即大约 157 个 batch。第二行 `157/157 [====` 表示已执行到第几个 batch。

#### 损失（代价 cost）
在 Course 1 里，我们学会通过监控代价来跟踪梯度下降的进展。理想情况下，代价会随算法迭代次数增加而下降。Tensorflow 把代价称为 `loss`。上面 `model.fit` 执行时，你看到每个 epoch 都显示 loss。[.fit](https://www.tensorflow.org/api_docs/python/tf/keras/Model) 方法返回包括 loss 在内的多种指标，被上面的 `history` 变量捕获。可以用它像下面这样把 loss 画成图来查看。

In [75]:
plot_loss_tf(history)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

#### 预测
要做预测，用 Keras 的 `predict`。下面，X[1015] 包含一张数字 2 的图。

In [76]:
image_of_two = X[1015]
display_digit(image_of_two)

prediction = model.predict(image_of_two.reshape(1,400))  # prediction

print(f" predicting a Two: \n{prediction}")
print(f" Largest Prediction index: {np.argmax(prediction)}")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

 predicting a Two: 
[[ -7.99  -2.23   0.77  -2.41 -11.66 -11.15  -9.53  -3.36  -4.42  -7.17]]
 Largest Prediction index: 2


最大的输出是 prediction[2]，表示预测的数字是 '2'。如果问题只需要一个选择，这就够了——用 NumPy 的 [argmax](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 选出它。如果问题需要概率，就需要一个 softmax：

In [77]:
prediction_p = tf.nn.softmax(prediction)

print(f" predicting a Two. Probability vector: \n{prediction_p}")
print(f"Total of predictions: {np.sum(prediction_p):0.3f}")

 predicting a Two. Probability vector: 
[[1.42e-04 4.49e-02 8.98e-01 3.76e-02 3.61e-06 5.97e-06 3.03e-05 1.44e-02
  5.03e-03 3.22e-04]]
Total of predictions: 1.000


要返回一个表示预测目标的整数，你需要最大概率的下标。这用 Numpy 的 [argmax](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 函数完成。

In [78]:
yhat = np.argmax(prediction_p)

print(f"np.argmax(prediction_p): {yhat}")

np.argmax(prediction_p): 2


我们对随机抽取的 64 个数字，比较预测值与标签。这要跑一会儿。

In [79]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
# You do not need to modify anything in this cell

m, n = X.shape

fig, axes = plt.subplots(8,8, figsize=(5,5))
fig.tight_layout(pad=0.13,rect=[0, 0.03, 1, 0.91]) #[left, bottom, right, top]
widgvis(fig)
for i,ax in enumerate(axes.flat):
    # Select random indices
    random_index = np.random.randint(m)
    
    # Select rows corresponding to the random indices and
    # reshape the image
    X_random_reshaped = X[random_index].reshape((20,20)).T
    
    # Display the image
    ax.imshow(X_random_reshaped, cmap='gray')
    
    # Predict using the Neural Network
    prediction = model.predict(X[random_index].reshape(1,400))
    prediction_p = tf.nn.softmax(prediction)
    yhat = np.argmax(prediction_p)
    
    # Display the label above the image
    ax.set_title(f"{y[random_index,0]},{yhat}",fontsize=10)
    ax.set_axis_off()
fig.suptitle("Label, yhat", fontsize=14)
plt.show()

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

我们看看其中一些错误。
>注意：增加训练 epoch 数可以消除这个数据集上的错误。

In [80]:
print( f"{display_errors(model,X,y)} errors out of {len(X)} images")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

15 errors out of 5000 images


### 恭喜！
你已经成功构建并使用了一个神经网络来做多分类。

<details>
  <summary><font size="2" color="darkgreen"><b>如果你想对其中的非评分代码做实验，请点击这里。</b></font></summary>
    <p><i><b>重要提示：请务必在你已经通过本作业后再这样做，以免影响自动评分器。</b></i>
    <ol>
        <li> 在 notebook 菜单，点击 “View” > “Cell Toolbar” > “Edit Metadata”</li>
        <li> 点击你想锁定/解锁的 code cell 旁边的 “Edit Metadata” 按钮</li>
        <li> 把 “editable” 属性值设为：
            <ul>
                <li> “true” 表示解锁 </li>
                <li> “false” 表示锁定 </li>
            </ul>
        </li>
        <li> 在 notebook 菜单，点击 “View” > “Cell Toolbar” > “None” </li>
    </ol>
    <p> 下面是上述步骤的一个简短演示：
        <br>
        <img src="https://lh3.google.com/u/0/d/14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" align="center" alt="unlock_cells.gif">
</details>